# 🤖 ROTBOTS — Full Pipeline
## Topic → Finished Video in One Notebook

---

Run every cell top to bottom. All 7 pipeline stages in one place.

**You don't need to understand the code.** Just fill in inputs and press ▶️ Play.

---

In [ ]:
# ============================================================
# SETUP — Run this cell once!
# ============================================================
!pip install -q requests beautifulsoup4 pymupdf edge-tts fal-client yt-dlp faster-whisper

import json, re, random, os, time, subprocess, shutil
from pathlib import Path
from IPython.display import display, Markdown, HTML, Audio, Video

from google.colab import drive
drive.mount('/content/drive')

BASE_DIR = Path('/content/drive/MyDrive/rotbots_workshop')
BASE_DIR.mkdir(parents=True, exist_ok=True)
TEMP = Path('/content/temp_work'); TEMP.mkdir(exist_ok=True)

print('✅ All dependencies installed')
print(f'📁 Workspace: {BASE_DIR}')

In [ ]:
# ============================================================
# API KEYS — Paste both keys here
# ============================================================

GROQ_API_KEY = ''     # Free: https://console.groq.com/keys
FAL_API_KEY = ''      # Video gen: https://fal.ai/dashboard/keys

# LLM Settings
LLM_MODEL = 'llama-3.3-70b-versatile'
LLM_TEMPERATURE = 0.8
LLM_MAX_TOKENS = 4096
GROQ_API_URL = 'https://api.groq.com/openai/v1/chat/completions'

if FAL_API_KEY: os.environ['FAL_KEY'] = FAL_API_KEY

if not GROQ_API_KEY: print('⚠️  Paste Groq key!')
elif not GROQ_API_KEY.startswith('gsk_'): print('⚠️  Key should start with gsk_')
else: print(f'✅ Groq: {LLM_MODEL}')
if FAL_API_KEY: print('✅ fal.ai ready')
else: print('⚠️  fal.ai key needed for video generation')


---
---
# 🎬 Part 1: Archive Scraper (Optional Found Footage)


This part is optional. Skip to Part 2 if you don't need archive footage.

---
## 📥 Enter Archive.org URLs

In [ ]:
ARCHIVE_URLS = [
    # 'https://archive.org/details/example_video',
]
PREFER_HEIGHT = 480
MAX_CLIP_DURATION = 10
MIN_CLIP_DURATION = 3
print(f'🎬 {len(ARCHIVE_URLS)} archive URL(s)')


---
---
# 📥 Part 2: Script Writer (Sources → Essay → Prompts)

This is the main starting point. Configure your topic and sources below.

In [ ]:
TOPIC = 'The history and ethics of AI-generated art'
SOURCES = ['https://en.wikipedia.org/wiki/Artificial_intelligence_art']
TARGET_VIDEO_LENGTH = 30
SECONDS_PER_SCENE = 5
SESSION_NAME = ''

if not SESSION_NAME:
    words = re.sub(r'[^a-zA-Z0-9\s]', '', TOPIC.lower()).split()
    SESSION_NAME = '-'.join(words[:4])
SESSION_DIR = BASE_DIR / SESSION_NAME
SESSION_DIR.mkdir(parents=True, exist_ok=True)
(SESSION_DIR / 'videos').mkdir(exist_ok=True)
(SESSION_DIR / 'audio').mkdir(exist_ok=True)
VIDEOS_DIR = SESSION_DIR / 'videos'
AUDIO_DIR = SESSION_DIR / 'audio'
ARCHIVE_DIR = SESSION_DIR / 'archive_clips'; ARCHIVE_DIR.mkdir(exist_ok=True)
sub_file = SESSION_DIR / 'subtitles.ass'
session_info = {'name': SESSION_NAME, 'topic': TOPIC, 'target_length': TARGET_VIDEO_LENGTH}
with open(SESSION_DIR / 'session_info.json', 'w') as f: json.dump(session_info, f, indent=2)
num_content_scenes = max(2, TARGET_VIDEO_LENGTH // SECONDS_PER_SCENE)
NUM_CHAPTERS = max(1, min(3, num_content_scenes // 2))
SECTIONS_PER_CHAPTER = max(1, num_content_scenes // NUM_CHAPTERS)
WORDS_PER_SCENE = SECONDS_PER_SCENE * 2.5
print(f'📂 Session: {SESSION_NAME}')
print(f'🎬 {TARGET_VIDEO_LENGTH}s = {num_content_scenes} scenes × {SECONDS_PER_SCENE}s')

*The rest of the pipeline continues below. Run each cell in order.*

*Generated by build_combined.py from 7 individual notebooks.*